# Monte Carlo Policy Gradient (MCPG)

MCPG searches for a best binary vector. This section states that search problem, then describes the ideal answer as a distribution $q^*$.

## 1. Problem setup

MCPG solves the binary optimization problem

$$
\min_{x \in \{-1,+1\}^{n}} f(x).
$$

The unknown is a vector $x = (x_1,\ldots,x_n)$ whose entries are restricted to $\{-1,+1\}$. The function $f$ gives the cost of each such vector. A solution is optimal when its cost equals the minimum value $f^{*}$.

### The ideal distribution $q^{*}$

Let $X^{*}$ be the set of all optimal solutions,

$$
X^{*} = \{\, x \in \{-1,+1\}^{n} : f(x) = f^{*} \,\}.
$$

The ideal sampler draws only from that set, and it draws each optimal vector with the same probability:

$$
q^{*}(x) =
\begin{cases}
1/|X^{*}| & \text{if } x \in X^{*}, \\
0 & \text{if } x \notin X^{*}.
\end{cases}
$$

One sample from $q^{*}$ is a minimizer of $f$. The distribution cannot be used directly: constructing it requires $X^{*}$, which is the set the algorithm is trying to find.

### The Gibbs distribution $q_{\lambda}$

The paper approaches $q^{*}$ with the Gibbs distribution at temperature $\lambda > 0$,

$$
q_{\lambda}(x) = \frac{1}{Z_{\lambda}} \exp\!\left(-\frac{f(x)}{\lambda}\right),
\qquad
Z_{\lambda} = \sum_{x \in \{-1,+1\}^{n}} \exp\!\left(-\frac{f(x)}{\lambda}\right).
$$

Vectors with smaller cost receive larger probability. As $\lambda \to 0$, $q_{\lambda}(x)$ converges to $q^{*}(x)$ for every $x$, so $q_{\lambda}$ is an approximation of $q^{*}$.

The normalizer $Z_{\lambda}$ adds one term for every binary vector, which is $2^{n}$ terms. Computing that sum is as hard as enumerating the whole search space, so $q_{\lambda}$ is not a practical sampler either.

### The trainable policy $p_{\theta}$

MCPG replaces $q_{\lambda}$ by a parameterized distribution $p_{\theta}$. The parameter $\theta$ is what training adjusts. The policy is built so that samples can be drawn without computing $Z_{\lambda}$, and the training objective pushes $p_{\theta}$ toward $q_{\lambda}$. Those samples are the candidate solutions of $\min f(x)$.

## 2 $q^{*}$ and $q_{\lambda}$
We have noticed that $q^{*}$ is the ideal distrbution where only the optimal $x\in X^{*}$ have the uniform probability on $q^{*}$. 

$$
q_{\lambda}(x) = \frac{1}{Z_{\lambda}} \exp\!\left(-\frac{f(x)}{\lambda}\right),
\qquad
Z_{\lambda} = \sum_{x \in \{-1,+1\}^{n}} \exp\!\left(-\frac{f(x)}{\lambda}\right).
$$

How $q^{*}$ = $q_{\lambda}$ as $\lambda \to 0$:

$$
q_{\lambda}(x) = \frac{\exp\!\left(-\frac{f(x)}{\lambda}\right)}{\sum_{x \in \{-1,+1\}^{n}} \exp\!\left(-\frac{f(x)}{\lambda}\right)}
\qquad
$$

Let $f^{*}$ be the optimal min value of $f(x)$, $B_{n} = \{-1,+1\}^{n}$

multiply $\frac{\exp{\frac{f^{*}}{\lambda}}}{exp{\frac{f^{*}}{\lambda}}}$ to $q_{\lambda}(x)$

$$
q_{\lambda}(x) = \frac{\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)}{\sum_{x \in B_n} \exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)}
\qquad
$$

For $x \in X^*$, $f(x) = f^*$ thus $\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)$ = $\exp0 = 1$. So we can pull out these $x\in X^*$.

$$
q_{\lambda}(x) = \frac{\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)}{|X^*| +  \sum_{x \in B_n \setminus X^*} \exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)}
\qquad
$$

Let $\lambda \to 0$, lets look at $\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)$. 
Noticed that $f^*$ is this min value of $f(x)$ thus $f^* \leq f(x) \to f^* - f(x) \leq 0$
For cases $x \in B_n \setminus X^*$, $f^* - f(x) <0$ thus $\exp\!\left(\frac{-\epsilon}{\lambda}\right)$ where $\epsilon > 0$ since $\lambda \to 0^+$ so $\frac{-\epsilon}{\lambda} \to -\infty$ therefore $\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right) \to 0$

Thus for $x \in B_n \setminus X^*$, $q_{\lambda \to 0^+}(x) = \frac{0}{|X^*| + 0} = 0$
For cases $x \in X^*$, $q_{\lambda \to 0^+}(x) = \frac{1}{|X^*| + 0} = \frac{1}{|X^*|}$

So we can conclued that as $\lambda \to 0^+, q^* = q_{\lambda}$.



## 3 $p_{\theta}(x)$

Why we need $p_{\theta}(x)$ instead of $q_{\lambda}$?  
Look at the term $Z_{\lambda}$ in $q_{\lambda}$: 
$$Z_{\lambda} = \sum_{x \in \{-1,+1\}^{n}} \exp\!\left(-\frac{f(x)}{\lambda}\right)$$
To compute this term requires look at every possiable $x$ which there are $2^n$ possiable $x$. This means the cost for computing $Z_{\lambda}$ is $O(2^n)$ since it increased exponentially is hard to sample. 

So we need to train $\theta$ so that $p_{\theta}$ approximately $q_{\lambda}$.

$p_{\theta}$ is defined as:
$$
p_{\theta}(x) = \prod_{i=1}^{n} \mu_i^{\frac{1+x_i}{2}} \, (1-\mu_i)^{\frac{1-x_i}{2}},
\qquad
\mu_i = \frac{1}{1 + e^{-\theta_i}}
$$

Let understand this equation step by step:

$x$ = $\{x_1, x_2,...,x_n\}$

$u_i$ = $p(x_i = +1)$ ==> when $x_i = 1$ $\frac{1+x_i}{2} = 1$, $\frac{1-x_i}{2} = 0$ so it keep $u_i$

$1-u_i$ = $p(x_i = -1)$ ==> when $x_i = -1$ $\frac{1+x_i}{2} = 0$, $\frac{1-x_i}{2} = 1$ so it keep $1 -u_i$

For example $u_i = 0.3$, $x = \{+1,-1,+1,-1\}$

$p_{\theta}(x) = p(x_1 = +1)p(x_2 = -1)p(x_3 = +1)p(x_4 = -1) ==> 0.3 * 0.7 * 0.3 * 0.7$

Each factor $\mu_i^{\frac{1+x_i}{2}} (1-\mu_i)^{\frac{1-x_i}{2}}$ is a Bernoulli distribution for one spin.
Writing $p_{\theta}$ as a product means we assume the spins $x_i$ are independent of each other.
This assumption is called the **mean field (MF) approximation**.

What if we do not use MF? Any distribution can be written with the chain rule:

$$
p(x) = p(x_1)\, p(x_2 \mid x_1)\, p(x_3 \mid x_1, x_2) \cdots p(x_n \mid x_1, \dots, x_{n-1})
$$

The factor $p(x_k \mid x_1, \dots, x_{k-1})$ needs one probability for each of the $2^{k-1}$ possible values of the earlier spins,
so a general distribution needs $2^n - 1$ numbers in total. With MF we only need $n$ numbers, $\theta_1, \dots, \theta_n$.

## KL divergence